# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   
import time

import sys
sys.path.append('../')
import helper_functions as hf


/var/folders/hk/7jjnrbrj53n1t8_bmkhf0_k00000gn/T/ipykernel_42805/1711932296.py:5: DeprecationWarning: Please import from 'ax.generation_strategy.generation_strategy' instead of 'ax.modelbridge.generation_strategy'. The latter is deprecated and will be removed in a future release.
  from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy


# initialize the optimizer

In [2]:
ax_client_init_file_name = 'optimizer/optimizer_00.json'
ax_client = hf.optimizer_init()
ax_client

[INFO 07-18 10:01:53] ax.service.utils.instantiation: Inferred value type of ParameterType.STRING for parameter surf_1. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
/Users/zeqing/opt/anaconda3/envs/sdlnano_plot/lib/python3.11/site-packages/ax/service/utils/instantiation.py:258: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "surf_1". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return ChoiceParameter(
[INFO 07-18 10:01:53] ax.service.utils.instantiation: Inferred value type of ParameterType.STRING for parameter surf_2. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
/Users/zeqing/opt/anaconda3/envs/sdlnano_plot/lib/python3.11/site-packages/ax/service/utils/ins

AxClient(experiment=Experiment(drug_surfactant))

In [3]:
ax_client.save_to_json_file(ax_client_init_file_name)

In [4]:
warm_start_df = pd.read_csv('/Users/zeqing/Desktop/warm_start.csv')

# generate recommendations

In [5]:
def load_warm_start(ax_client, warm_start_df, objective_col="obj_surf_conc"):

    # reset index so trial indices go 0,1,2,...
    df = warm_start_df.reset_index(drop=True)

    # get the list of parameter names your experiment expects
    param_names = list(ax_client.experiment.search_space.parameters.keys())

    for trial_index, row in df.iterrows():
        # build the dict of parameter values for this trial
        parameterization = {name: row[name] for name in param_names}

        # attach it (positional arg!)
        ax_client.attach_trial(parameterization)

        # then mark it completed with your observed objective
        ax_client.complete_trial(trial_index, raw_data=row[objective_col])

    return ax_client


In [6]:
ax_client = load_warm_start(ax_client, warm_start_df, "obj_surf_conc")
ax_client

[INFO 07-18 10:01:59] ax.service.ax_client: Completed trial 0 with data: {'obj_surf_conc': (100.0, None)}.
[INFO 07-18 10:01:59] ax.service.ax_client: Completed trial 1 with data: {'obj_surf_conc': (95.0, None)}.
[INFO 07-18 10:01:59] ax.service.ax_client: Completed trial 2 with data: {'obj_surf_conc': (100.0, None)}.
[INFO 07-18 10:01:59] ax.service.ax_client: Completed trial 3 with data: {'obj_surf_conc': (100.0, None)}.
[INFO 07-18 10:01:59] ax.service.ax_client: Completed trial 4 with data: {'obj_surf_conc': (100.0, None)}.
[INFO 07-18 10:01:59] ax.service.ax_client: Completed trial 5 with data: {'obj_surf_conc': (91.0, None)}.
[INFO 07-18 10:01:59] ax.service.ax_client: Completed trial 6 with data: {'obj_surf_conc': (100.0, None)}.
[INFO 07-18 10:01:59] ax.service.ax_client: Completed trial 7 with data: {'obj_surf_conc': (100.0, None)}.
[INFO 07-18 10:01:59] ax.service.ax_client: Completed trial 8 with data: {'obj_surf_conc': (100.0, None)}.
[INFO 07-18 10:01:59] ax.service.ax_cli

AxClient(experiment=Experiment(drug_surfactant))

In [7]:
ax_client.get_trials_data_frame()


,trial_index,arm_name,trial_status,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1,surf_1_conc,surf_2,surf_2_conc,surf_3,surf_3_conc,drug_conc
0,0,0_0,COMPLETED,100.0,0.2063,0.3073,0.0373,s8,12,s6,39,s7,20,25.0
1,1,1_0,COMPLETED,95.0,0.2063,0.3073,0.0373,s3,51,s2,4,s6,40,25.0
2,2,2_0,COMPLETED,100.0,0.4045,0.4196,0.0728,s5,1,s7,56,s3,0,25.0
3,3,3_0,COMPLETED,100.0,0.4045,0.4196,0.0728,s3,28,s3,6,s8,14,25.0
4,4,4_0,COMPLETED,100.0,0.2962,0.4364,0.0493,s1,27,s4,63,s5,3,25.0
5,5,5_0,COMPLETED,91.0,0.2962,0.4364,0.0493,s6,40,s8,29,s7,22,25.0
6,6,6_0,COMPLETED,100.0,0.3528,0.2810,0.0711,s1,5,s1,24,s6,18,25.0
7,7,7_0,COMPLETED,100.0,0.3528,0.2810,0.0711,s6,10,s5,1,s1,36,25.0
8,8,8_0,COMPLETED,100.0,0.2063,0.3073,0.0373,s4,4,s4,15,s1,50,25.0
9,9,9_0,COMPLETED,86.0,0.2063,0.3073,0.0373,s3,37,s4,17,s5,32,25.0


In [8]:
ax_client.save_to_json_file('/Users/zeqing/0_Github/drug_surfactant/experiments/Exploration only_do not use/20250718_SAASBO/iteration_0/optimizer/optimizer_0_loaded.json')